> **Version étudiant** — les cellules d'exercice ne rappellent que la consigne : le code est à écrire entièrement par vous-même, sans squelette imposé. Un exemple travaillé sur un cas analogue précède toujours ce type d'exercice. Les cellules repérées par **Question** n'ont pas de correction automatique : exécutez le code fourni, observez, et répondez par écrit. La version corrigée est téléchargeable depuis la page du cours.

# Théorie statistique de l'apprentissage

**Notebook 2/9 — Introduction à l'apprentissage supervisé**
*L3 MIASHS → Master, Guillaume Metzler, Université Lyon 2*

Avant de comparer des algorithmes entre eux, il faut poser un cadre précis :
qu'est-ce qu'on cherche à apprendre, avec quoi le mesure-t-on, et pourquoi un
modèle qui « colle » parfaitement aux données d'entraînement peut se révéler
catastrophique sur de nouvelles données ? Ce notebook pose les bases
théoriques utilisées dans tout le reste du cours.

Ce notebook porte sur :
- le cadre général de l'apprentissage supervisé (régression, classification,
  représentation des données) ;
- le risque vrai, le risque empirique et la minimisation du risque empirique
  (ERM) ;
- le classifieur de Bayes et la notion de risque incompressible ;
- le compromis biais-variance et le phénomène de sur-apprentissage /
  sous-apprentissage ;
- l'intuition des bornes de généralisation et le rôle de la complexité de
  l'espace d'hypothèses.

La procédure expérimentale (découpage train / validation / test, validation
croisée) et les mesures de performance sont traitées dans le notebook
suivant.


In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from scipy import integrate
from scipy.stats import norm

from sklearn.datasets import (
    make_regression, make_classification, make_moons,
    load_iris, load_wine,
)
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score

RNG = np.random.RandomState(0)
plt.rcParams["figure.figsize"] = (6, 4)
print("Bibliothèques chargées.")


## 1. Le cadre de l'apprentissage supervisé

On dispose d'un échantillon de $m$ observations labellisées
$S = \{(x_i, y_i)\}_{i=1}^m$, tirées i.i.d. selon une distribution jointe
**inconnue** $D$ sur $\mathcal{X}\times\mathcal{Y}$. $\mathcal{X}\subset
\mathbb{R}^d$ est l'espace des descripteurs (features) ; la nature de
$\mathcal{Y}$ détermine le type de tâche :

- $\mathcal{Y}\subset\mathbb{R}$ : **régression** ;
- $\mathcal{Y}=\{-1,+1\}$ (ou $\{0,1\}$) : **classification binaire** ;
- $\mathcal{Y}=\{1,\dots,q\}$ avec $q>2$ : **classification multi-classes**.

On cherche une **hypothèse** $h:\mathcal{X}\to\mathcal{Y}$, apprise à partir
de $S$, qui généralise bien à de nouvelles observations $(x,y)\sim D$ jamais
vues pendant l'entraînement. Quelle que soit la nature brute des données
(images, texte, séries temporelles...), elles sont in fine représentées sous
forme d'une matrice $X\in\mathbb{R}^{m\times d}$ : une ligne par observation,
une colonne par descripteur.

In [ ]:

# Deux jeux de donnees jouets pour illustrer les deux familles de taches
X_reg, y_reg = make_regression(n_samples=150, n_features=1, noise=15.0, random_state=0)
X_clf, y_clf = make_classification(
    n_samples=200, n_features=2, n_informative=2, n_redundant=0,
    n_clusters_per_class=1, class_sep=1.5, random_state=0,
)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].scatter(X_reg[:, 0], y_reg, alpha=0.6, color="tab:blue")
axes[0].set_title("Régression : $y$ continu")
axes[0].set_xlabel(r"$x$")
axes[0].set_ylabel(r"$y$")

for classe, couleur in zip([0, 1], ["tab:orange", "tab:green"]):
    masque = y_clf == classe
    axes[1].scatter(X_clf[masque, 0], X_clf[masque, 1], alpha=0.7, color=couleur,
                     label=f"classe {classe}")
axes[1].set_title("Classification binaire : $y \\in \\{0,1\\}$")
axes[1].set_xlabel(r"$x_1$")
axes[1].set_ylabel(r"$x_2$")
axes[1].legend()

plt.tight_layout()
plt.show()

print("X_reg :", X_reg.shape, " -- y_reg :", y_reg.shape)
print("X_clf :", X_clf.shape, " -- y_clf :", y_clf.shape)


On peut représenter de la même façon un vrai jeu de données. Prenons
`load_iris` (classification de fleurs en 3 espèces selon 4 mesures) :

In [ ]:

X_iris, y_iris = load_iris(return_X_y=True)
print(f"{X_iris.shape[0]} observations, {X_iris.shape[1]} descripteurs, "
      f"{len(np.unique(y_iris))} classes")


### Exercice 1 — Représentation des données

Faites la même chose avec `load_wine` (classification de vins en 3 classes
selon leurs caractéristiques chimiques) : combien d'observations $m$, de
descripteurs $d$, de classes ? Concluez, dans un `print`, sur le type de
tâche (régression, classification binaire ou multi-classes).

In [ ]:
# Chargez load_wine (return_X_y=True), affichez m, d et le nombre de classes, puis concluez dans un print sur le type de tache.


## 2. Risque, risque empirique et minimisation du risque empirique

Pour mesurer l'erreur commise par $h$ sur une observation, on se donne une
**fonction de perte** $\ell:\mathcal{X}\times\mathcal{Y}\to\mathbb{R}_+$. On
voudrait minimiser le **risque vrai** (True Risk) :

$$R^\ell(h) = \mathbb{E}_{(x,y)\sim D}\big[\ell(h(x), y)\big].$$

Mais $D$ est inconnue : ce risque n'est **pas calculable** en pratique. On
minimise donc plutôt le **risque empirique** (Empirical Risk), calculé sur
l'échantillon disponible $S$ :

$$R_S^\ell(h) = \frac{1}{m}\sum_{i=1}^m \ell(h(x_i), y_i).$$

Pour une tâche de classification, la perte la plus naturelle est la
**perte 0-1**, $\ell(h(x),y) = \mathbb{1}_{\{h(x)\neq y\}}$ : le risque
empirique associé est alors simplement le taux d'erreur sur $S$. Pour une
régression, on utilise typiquement la perte absolue ou la perte
quadratique.

In [ ]:

# Risque empirique "a la main" sur un petit echantillon jouet

# Classification (perte 0-1)
y_true_clf = np.array([1, 0, 1, 1, 0, 0, 1, 0])
y_pred_clf = np.array([1, 0, 0, 1, 0, 1, 1, 0])
risque_clf = np.mean(y_true_clf != y_pred_clf)
print(f"Risque empirique (perte 0-1)      : {risque_clf:.3f}")

# Regression (perte absolue)
y_true_reg = np.array([2.0, 3.0, 1.5, 5.0])
y_pred_reg = np.array([2.1, 3.5, 1.0, 4.0])
risque_reg = np.mean(np.abs(y_true_reg - y_pred_reg))
print(f"Risque empirique (perte absolue)  : {risque_reg:.3f}")


### Exercice 2 — Risque empirique, deux autres pertes

Calculez le risque empirique avec la **perte quadratique** sur
`y_true_q = [3.0, -1.0, 2.0, 0.5, 5.0]` et
`y_pred_q = [2.5, -1.5, 2.0, 1.0, 4.0]`, puis avec la **perte 0-1** sur
`y_true_b = [1, 1, 0, 1, 0, 0, 1, 1]` et
`y_pred_b = [1, 0, 0, 1, 0, 1, 1, 1]`.

In [ ]:
# Calculez le risque empirique (perte quadratique) sur y_true_q/y_pred_q, puis (perte 0-1) sur y_true_b/y_pred_b.


Le principe de la **minimisation du risque empirique** (ERM,
*Empirical Risk Minimization*) consiste, pour un espace d'hypothèses $H$
fixé à l'avance, à choisir :

$$h_S^\star = \arg\min_{h\in H} R_S^\ell(h).$$

Rien ne garantit a priori que $h_S^\star$ minimise aussi le risque vrai
$R^\ell$ : c'est tout l'enjeu des sections suivantes. Voyons d'abord le
mécanisme sur un espace d'hypothèses volontairement très simple, pour bien
voir ce que fait concrètement l'ERM.

In [ ]:

# ERM sur H = { classifieurs a seuil sur x }, en classification 1D
X_H, y_H = make_classification(
    n_samples=40, n_features=1, n_informative=1, n_redundant=0,
    n_clusters_per_class=1, class_sep=1.2, random_state=3,
)
x_H = X_H.ravel()

seuils = np.linspace(x_H.min(), x_H.max(), 25)
risques_seuils = [np.mean((x_H > s).astype(int) != y_H) for s in seuils]

meilleur_seuil = seuils[int(np.argmin(risques_seuils))]
print(f"Meilleur seuil (ERM) : {meilleur_seuil:.3f}, "
      f"risque empirique = {min(risques_seuils):.3f}")

plt.figure(figsize=(6.5, 4))
plt.plot(seuils, risques_seuils, "o-")
plt.axvline(meilleur_seuil, color="tab:red", linestyle="--", label="seuil ERM")
plt.xlabel(r"seuil $s$ (hypothèse : $h(x) = \mathbb{1}_{\{x>s\}}$)")
plt.ylabel("Risque empirique")
plt.title(r"ERM sur $H$ = {classifieurs à seuil}")
plt.legend()
plt.show()


### Exercice 3 — ERM sur un espace de constantes (régression)

Générez `x_r, y_r = make_regression(n_samples=50, n_features=1, noise=10.0,
random_state=7)` (aplatissez `x_r`). On prend pour $H$ l'ensemble des 20
hypothèses **constantes** régulièrement espacées entre le min et le max de
`y_r`. Calculez le risque empirique (perte quadratique) de chacune,
affichez celle qui le minimise ainsi que sa valeur, et tracez le risque
empirique en fonction de la constante candidate.

In [ ]:
# Sur make_regression, testez 20 hypotheses constantes entre min(y_r) et max(y_r), calculez leur risque empirique (perte quadratique), affichez la meilleure et tracez la courbe risque empirique / constante.


### Le classifieur de Bayes

Même avec un espace d'hypothèses $H$ aussi riche que l'on veut et une
infinité de données, il existe une limite : le **classifieur de Bayes**
$h^\star(x) = \arg\max_y \Pr(y\mid x)$, celui qui minimise le risque vrai
sur *toutes* les fonctions possibles. Son risque, le **risque de Bayes**
$R^\star$, n'est pas nul dès que les classes se chevauchent : c'est une
erreur incompressible, propre à la distribution des données, indépendante
du modèle choisi.

In [ ]:

# Deux classes gaussiennes qui se chevauchent
mu0, mu1, sigma = -1.0, 1.0, 1.0

x_grille = np.linspace(-6, 6, 400)
dens0 = norm.pdf(x_grille, mu0, sigma)
dens1 = norm.pdf(x_grille, mu1, sigma)

plt.figure(figsize=(6.5, 4))
plt.plot(x_grille, dens0, label="classe 0", color="tab:blue")
plt.plot(x_grille, dens1, label="classe 1", color="tab:orange")
plt.fill_between(x_grille, np.minimum(dens0, dens1), color="gray", alpha=0.4,
                  label="zone de recouvrement")
plt.legend()
plt.title("Deux classes gaussiennes qui se chevauchent")
plt.xlabel("x")
plt.show()

# Risque de Bayes theorique = aire ponderee par les priors sous le min des deux densites
risque_bayes, _ = integrate.quad(
    lambda x: 0.5 * min(norm.pdf(x, mu0, sigma), norm.pdf(x, mu1, sigma)), -20, 20,
)
print(f"Risque de Bayes (théorique) : {risque_bayes:.4f}")

# On simule un gros echantillon et on entraine un classifieur
X_bayes = np.concatenate([
    RNG.normal(mu0, sigma, size=20000), RNG.normal(mu1, sigma, size=20000),
]).reshape(-1, 1)
y_bayes = np.concatenate([np.zeros(20000), np.ones(20000)])
X_tr_b, X_te_b, y_tr_b, y_te_b = train_test_split(X_bayes, y_bayes, test_size=0.3, random_state=0)

clf_bayes = LogisticRegression().fit(X_tr_b, y_tr_b)
erreur_test = 1 - accuracy_score(y_te_b, clf_bayes.predict(X_te_b))
print(f"Erreur de test du classifieur entraîné : {erreur_test:.4f}")


$$ $$

**Question 1 :** Que représente, sur le graphique, la zone grisée où les deux densités se chevauchent ? Que vaut-elle numériquement ?

$$ $$

$$ $$

**Question 2 :** Le classifieur entraîné obtient une erreur de test proche du risque de Bayes théorique. Pourrait-on faire mieux, même avec un classifieur beaucoup plus complexe et une infinité de données d'entraînement ? Pourquoi ?

$$ $$

## 3. Sur-apprentissage, sous-apprentissage et compromis biais-variance

Un espace $H$ trop pauvre ne peut pas approcher la vraie fonction sous-jacente :
c'est le **sous-apprentissage** (*underfitting*), un modèle au biais élevé.
Un espace $H$ trop riche colle aux données d'entraînement, y compris à leur
bruit : c'est le **sur-apprentissage** (*overfitting*), un modèle à la
variance élevée. On l'illustre classiquement en faisant varier la
complexité du modèle et en comparant risque empirique et risque de
test.

In [ ]:

# Regression non lineaire bruitee, ajustee par des polynomes de degre croissant
def f_vraie(x):
    return 0.5 * x**3 - 2 * x**2 + x + 5

rng_poly = np.random.RandomState(0)
x_pop = np.sort(rng_poly.uniform(-3, 3, size=100))
sigma_bruit = 5.0
y_pop = f_vraie(x_pop) + rng_poly.normal(scale=sigma_bruit, size=len(x_pop))

X_poly = x_pop.reshape(-1, 1)
X_tr_p, X_te_p, y_tr_p, y_te_p = train_test_split(X_poly, y_pop, test_size=0.3, random_state=0)

degres = np.arange(1, 15)
erreurs_train, erreurs_test = [], []
for deg in degres:
    modele = make_pipeline(PolynomialFeatures(deg), StandardScaler(), LinearRegression())
    modele.fit(X_tr_p, y_tr_p)
    erreurs_train.append(np.mean((y_tr_p - modele.predict(X_tr_p)) ** 2))
    erreurs_test.append(np.mean((y_te_p - modele.predict(X_te_p)) ** 2))

plt.figure(figsize=(7, 4.5))
plt.plot(degres, erreurs_train, "o-", label="risque empirique (entraînement)")
plt.plot(degres, erreurs_test, "o-", label="risque de test")
plt.yscale("log")
plt.xlabel("Degré du polynôme (complexité)")
plt.ylabel("MSE (échelle log)")
plt.title("Risque empirique et risque de test selon la complexité")
plt.legend()
plt.show()

print("Degré minimisant le risque de test :", degres[int(np.argmin(erreurs_test))])


$$ $$

**Question 3 :** Que se passe-t-il pour le risque empirique quand le degré du polynôme augmente ? Est-ce surprenant ?

$$ $$

$$ $$

**Question 4 :** Le risque de test, lui, ne suit pas la même trajectoire. Décrivez sa forme et expliquez pourquoi il remonte au-delà d'un certain degré.

$$ $$

### Exercice 4 — Sur/sous-apprentissage avec un arbre de décision

On reprend la même mécanique, mais pour une tâche de **classification**,
avec `max_depth` d'un `DecisionTreeClassifier` comme paramètre de
complexité. Générez `make_classification` (200 exemples, 2 descripteurs
informatifs, `flip_y=0.15`), séparez en train/test (70 % / 30 %), puis pour
`max_depth` de 1 à 15 calculez l'accuracy train et test. Tracez les deux
courbes et affichez la profondeur qui maximise l'accuracy de test.

In [ ]:
# make_classification (200, flip_y=0.15) puis train/test 70/30. Pour max_depth de 1 a 15, accuracy train/test d'un DecisionTreeClassifier. Tracez les deux courbes et affichez la profondeur optimale.


On peut formaliser ce compromis. Si les données sont générées selon
$y = f(x) + \varepsilon$ avec $\mathbb{E}[\varepsilon]=0$, et que $h$ est
appris sur un échantillon $S$ (donc une quantité aléatoire), l'erreur
quadratique de généralisation se décompose en trois termes :

$$\mathbb{E}[(y-h(x))^2] = \underbrace{(\mathbb{E}[h(x)]-f(x))^2}_{\text{biais}^2}
+ \underbrace{\mathbb{E}\big[(\mathbb{E}[h(x)]-h(x))^2\big]}_{\text{variance}}
+ \underbrace{\mathbb{E}[(y-f(x))^2]}_{\text{erreur de Bayes (incompressible)}}.$$

Un modèle simple a un biais élevé et une variance faible ; un modèle
complexe, l'inverse. Le même compromis s'observe directement en comparant
plusieurs classifieurs de complexité croissante sur les mêmes données.

In [ ]:

# Comparaison visuelle : k-NN, complexite decroissante quand k augmente
X_knn, y_knn = make_moons(n_samples=220, noise=0.28, random_state=0)
X_tr_k, X_te_k, y_tr_k, y_te_k = train_test_split(X_knn, y_knn, test_size=0.3, random_state=0)

xx, yy = np.meshgrid(
    np.linspace(X_knn[:, 0].min() - 0.5, X_knn[:, 0].max() + 0.5, 200),
    np.linspace(X_knn[:, 1].min() - 0.5, X_knn[:, 1].max() + 0.5, 200),
)

k_values = [1, 5, 15, 60]
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, k in zip(axes, k_values):
    clf_k = KNeighborsClassifier(n_neighbors=k).fit(X_tr_k, y_tr_k)
    zz = clf_k.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, zz, alpha=0.3, cmap="coolwarm")
    ax.scatter(X_tr_k[:, 0], X_tr_k[:, 1], c=y_tr_k, cmap="coolwarm", edgecolor="k", s=15)
    acc_tr = accuracy_score(y_tr_k, clf_k.predict(X_tr_k))
    acc_te = accuracy_score(y_te_k, clf_k.predict(X_te_k))
    ax.set_title(f"k={k}\ntrain={acc_tr:.2f}  test={acc_te:.2f}")
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()
plt.show()


$$ $$

**Question 5 :** Comment évolue la frontière de décision quand $k$ augmente ? Reliez cette observation au compromis biais-variance : quel(s) k correspond(ent) plutôt à une variance élevée, lequel à un biais élevé ?

$$ $$

In [ ]:

# Estimation numerique du biais^2 et de la variance par bootstrap (regression)
rng_bv = np.random.RandomState(1)
x_bv = np.sort(rng_bv.uniform(-3, 3, size=40))
sigma_bv = 6.0

def biais_variance(degre, n_boot=200):
    preds = np.zeros((n_boot, len(x_bv)))
    for b in range(n_boot):
        idx = rng_bv.randint(0, len(x_bv), size=len(x_bv))
        x_b = x_bv[idx]
        y_b = f_vraie(x_b) + rng_bv.normal(scale=sigma_bv, size=len(idx))
        modele_b = make_pipeline(PolynomialFeatures(degre), StandardScaler(), LinearRegression())
        modele_b.fit(x_b.reshape(-1, 1), y_b)
        preds[b] = modele_b.predict(x_bv.reshape(-1, 1))
    pred_moyenne = preds.mean(axis=0)
    biais2 = np.mean((pred_moyenne - f_vraie(x_bv)) ** 2)
    variance = np.mean(preds.var(axis=0))
    return biais2, variance

for degre in [1, 9]:
    b2, v = biais_variance(degre)
    print(f"degré={degre:2d}  ->  biais²={b2:8.2f}   variance={v:8.2f}   "
          f"biais²+variance={b2+v:8.2f}")
print(f"(bruit théorique sigma² = {sigma_bv**2:.1f}, terme incompressible)")


On retrouve numériquement l'intuition attendue : le polynôme de degré 1
(trop simple) a un biais élevé et une variance faible, tandis que le
polynôme de degré 9 (trop riche pour seulement 40 points) a un biais quasi
nul mais une variance qui explose. La somme biais² + variance suit la même
logique en U que le risque de test observé plus haut.

## 4. Complexité d'un espace d'hypothèses et bornes de généralisation

Peut-on quantifier, avant même de voir de nouvelles données, l'écart entre
risque vrai et risque empirique ? Une **borne de généralisation** (souvent
appelée borne PAC, *Probably Approximately Correct*) a la forme générale :

$$\Pr\big(|R^\ell(h) - R_S^\ell(h)| \ge \varepsilon\big) \le \delta.$$

Quand $H$ est de taille finie, une borne de ce type (obtenue via l'inégalité
de Hoeffding et l'union bound) s'écrit : avec probabilité au moins
$1-\delta$, pour toute hypothèse $h\in H$,

$$R^\ell(h) \le R_S^\ell(h) + \sqrt{\frac{\ln|H| + \ln(2/\delta)}{2m}}.$$

Le terme correctif **croît avec la complexité** $|H|$ et **décroît avec la
taille de l'échantillon** $m$ (vitesse en $O(1/\sqrt{m})$). Quand $H$ est de
taille infinie (par exemple l'ensemble des classifieurs linéaires), on
utilise d'autres mesures de complexité, comme la dimension de
Vapnik-Chervonenkis (VC) ou la complexité de Rademacher, qui mesure
informellement la capacité de $H$ à s'ajuster au bruit du jeu de données.
Le principe reste le même : plus $H$ est riche, plus la borne se dégrade ;
plus $m$ est grand, plus elle se resserre.

In [ ]:

# Terme correctif de la borne, en fonction de m, pour plusieurs |H|
delta = 0.05
tailles_H = [10, 100, 1000]
m_grid = np.arange(50, 5000, 50)

plt.figure(figsize=(7, 4.5))
for taille in tailles_H:
    borne = np.sqrt((np.log(taille) + np.log(2 / delta)) / (2 * m_grid))
    plt.plot(m_grid, borne, label=f"|H| = {taille}")
plt.xlabel("Taille de l'échantillon $m$")
plt.ylabel(r"$\sqrt{(\ln|H|+\ln(2/\delta))/(2m)}$")
plt.title("Borne de généralisation (H fini) selon $m$")
plt.legend()
plt.show()


### Exercice 5 — Borne de généralisation selon $\delta$

Sur le même principe que ci-dessus, avec $|H|=500$ fixé, tracez le terme
correctif de la borne pour $\delta \in \{0.2, 0.05, 0.01\}$, et affichez sa
valeur numérique pour $m=1000$ dans chacun des trois cas.

In [ ]:
# |H|=500 fixe. Tracez le terme correctif de la borne pour delta dans {0.2, 0.05, 0.01} en fonction de m, et affichez sa valeur pour m=1000 dans chaque cas.


La borne théorique prédit que l'écart entre risque vrai et risque
empirique se resserre quand $m$ augmente. Vérifions-le empiriquement, sur
un modèle et des données fixés.

In [ ]:

# Ecart risque empirique / risque de test selon la taille m d'entrainement
X_pool, y_pool = make_classification(
    n_samples=20000, n_features=10, n_informative=6, flip_y=0.1, random_state=0,
)
X_train_pool, X_test_m, y_train_pool, y_test_m = train_test_split(
    X_pool, y_pool, test_size=5000, random_state=0,
)

tailles_m = [20, 50, 100, 300, 1000, 3000, 10000]
risques_train, risques_test, ecarts = [], [], []
for m in tailles_m:
    X_m, y_m = X_train_pool[:m], y_train_pool[:m]
    modele_m = LogisticRegression(max_iter=2000).fit(X_m, y_m)
    r_train = 1 - accuracy_score(y_m, modele_m.predict(X_m))
    r_test = 1 - accuracy_score(y_test_m, modele_m.predict(X_test_m))
    risques_train.append(r_train)
    risques_test.append(r_test)
    ecarts.append(r_test - r_train)

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(tailles_m, risques_train, "o-", label="risque empirique (entraînement)")
ax.plot(tailles_m, risques_test, "o-", label="risque sur un grand test fixe (proxy du risque vrai)")
ax.set_xscale("log")
ax.set_xlabel("Taille de l'échantillon d'entraînement $m$ (échelle log)")
ax.set_ylabel("Risque (1 − accuracy)")
ax.set_title("Régression logistique : risque empirique et risque de test selon $m$")
ax.legend()
plt.show()

print("Écarts (risque test − risque empirique) :", np.round(ecarts, 3))


$$ $$

**Question 6 :** Comment évolue l'écart entre risque empirique et risque de test quand $m$ augmente ? Ce comportement est-il cohérent avec la forme de la borne de généralisation vue plus haut ?

$$ $$

### Exercice 6 (niveau Master) — Effet de la complexité sur l'écart risque
empirique / risque de test

Reprenez le principe de la démonstration précédente (grand ensemble de test
fixe, taille $m$ d'entraînement variable) mais avec un
`DecisionTreeClassifier(max_depth=None)` à la place de la régression
logistique. Tracez les deux courbes et comparez, dans un `print`, l'écart
obtenu à la plus petite taille $m$ à celui obtenu avec la régression
logistique.

In [ ]:
# Meme experience que ci-dessus (X_train_pool, y_train_pool, X_test_m, y_test_m deja definis) mais avec un DecisionTreeClassifier(max_depth=None). Tracez les deux courbes et comparez l'ecart au plus petit m a celui de la regression logistique (variable ecarts).


## Pour aller plus loin

Nous avons vu le vocabulaire et les résultats qui structurent tout le
reste du cours : risque vrai, risque empirique, ERM, classifieur de Bayes,
compromis biais-variance, et le rôle conjoint de la complexité de $H$ et de
la taille de l'échantillon dans les bornes de généralisation. Le notebook
suivant s'appuie sur ces notions pour construire une procédure
expérimentale rigoureuse (train / validation / test, validation croisée) et
choisir les bonnes mesures de performance avant de comparer de vrais
algorithmes entre eux.